# LoRA verifier training (v12) — Team Outliers

QLoRA-tunes **Qwen2.5-14B-Instruct** to answer the exact Yes/No verification prompts used at
inference. Training data: the 299 labelled sample rows (repeated 4x) + the team's 5,000-row
synthetic set (public Kaggle dataset — declare it on the Discussion tab).

**Run:** internet ON, GPU T4×2, attach: competition data + the synthetic dataset. ~3–5 h.
**Then:** Save Version → Output tab → create a (can be private) dataset from `lora_adapter/` →
attach that dataset to `kaggle_inference_v12.ipynb`.

Rulebook: fine-tuning pretrained models is permitted (not on the test set — none is used here);
the training notebook is optional for Phase 2 but we keep it Kaggle-runnable anyway.


In [ ]:
%pip install -q -U peft bitsandbytes accelerate
import os, re, glob, json, random
import numpy as np, pandas as pd, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
BASE = "Qwen/Qwen2.5-14B-Instruct"
MAXLEN, EPOCHS, LR, REAL_REPEAT = 1024, 2, 1e-4, 4


In [ ]:
BN2EN = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
MATH_KW = re.compile(
    r"শতকরা|সম্ভাবনা|গুণিতক|মৌলিক|ধারাটি|সমষ্টি|যোগফল|বর্গ(?:মূল|ইঞ্চি|ফুট)?|"
    r"ক্ষতি হ|লাভ হ|বয়স|কত ?দিনে|গড়|অনুপাত|ভগ্নাংশ|সুদ|আসল|"
    r"[0-9০-৯]\s*[+\-*/=]|√|x\s*[*+=]|\*\*|%")

def has_context(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return False
    return str(v).strip() not in ("", "[NULL]", "NULL", "nan")

def route_row(ctx, q):
    if MATH_KW.search(str(q)): return "math"
    return "grounded" if has_context(ctx) else "closedbook"

def _clip(s, n):
    s = str(s) if s is not None else ""
    return s if len(s) <= n else s[:n] + "…"

SYS = ("You are a meticulous bilingual (Bengali/English) fact-checker. You judge whether a "
       "Bengali answer is factually correct and, when a passage is given, supported by it.")
TRAP_RULES = (
    "Rules:\n"
    "- The response is faithful ONLY if it answers exactly what the question asks AND is factually correct.\n"
    "- Copying words from the passage does NOT make it faithful: if the question asks for a year, a name "
    "is wrong; if the question's premise is not supported by the passage, be skeptical.\n"
    "- Partially wrong, wrong entity/date/number, or fabricated details => hallucinated.\n")

def lp_prompt(ctx, q, r):
    """Single-token Yes/No verification prompt (evidence-free; same for train + LoRA inference)."""
    if has_context(ctx):
        return (f"Passage (Bengali):\n{_clip(ctx, 1400)}\n\nQuestion (Bengali): {_clip(q, 500)}\n"
                f"Response (Bengali): {_clip(r, 700)}\n\n" + TRAP_RULES +
                "Is the response faithful? Answer with exactly one word: Yes or No.")
    return ("Judge using your own knowledge of Bengali/Bangladesh facts.\n"
            f"Question (Bengali): {_clip(q, 500)}\nProposed answer (Bengali): {_clip(r, 700)}\n"
            "Is the proposed answer factually correct? Answer with exactly one word: Yes or No.\nVerdict:")


def find_labeled():
    for p in sorted(glob.glob("/kaggle/input/**/*", recursive=True)):
        b = os.path.basename(p).lower()
        if b.replace(" ", "").startswith("train") and b.endswith(".csv"): return p
    for p in sorted(glob.glob("/kaggle/input/**/*.json", recursive=True)):
        if "sample" in os.path.basename(p).lower() and "submission" not in p.lower(): return p
    return None

def find_synth():
    for p in sorted(glob.glob("/kaggle/input/**/synthetic*train*.csv", recursive=True)): return p
    for p in sorted(glob.glob("/kaggle/input/**/synthetic*.csv", recursive=True)): return p
    return None

def load_pairs(path):
    df = pd.read_json(path) if path.endswith(".json") else pd.read_csv(path)
    low = {c.lower().strip(): c for c in df.columns}
    out = []
    for i in range(len(df)):
        ctx = df[low.get("context")].iloc[i] if low.get("context") else None
        q, r = df[low["prompt_bn"]].iloc[i], df[low["response_bn"]].iloc[i]
        y = int(df[low["label"]].iloc[i])
        out.append((lp_prompt(ctx, q, r), " Yes" if y == 1 else " No"))
    return out

real_p, syn_p = find_labeled(), find_synth()
assert real_p, "attach the competition data"
pairs = load_pairs(real_p) * REAL_REPEAT
if syn_p:
    pairs += load_pairs(syn_p)
    print("synthetic:", syn_p)
else:
    print("WARNING: synthetic dataset not attached -> training on 299x%d only" % REAL_REPEAT)
random.shuffle(pairs)
n_val = max(64, len(pairs) // 50)
val_pairs, train_pairs = pairs[:n_val], pairs[n_val:]
print(f"train={len(train_pairs)}  val={len(val_pairs)}")


In [ ]:
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          Trainer, TrainingArguments)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None: tok.pad_token = tok.eos_token

def encode(pair):
    user, ans = pair
    prompt = tok.apply_chat_template(
        [{"role": "system", "content": SYS}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True)
    pi = tok(prompt, truncation=True, max_length=MAXLEN - 4, add_special_tokens=False)["input_ids"]
    ai = tok(ans, add_special_tokens=False)["input_ids"]
    ids = (pi + ai)[:MAXLEN]
    labels = [-100] * len(pi) + ai
    labels = labels[:MAXLEN]
    return {"input_ids": ids, "labels": labels, "attention_mask": [1] * len(ids)}

class DS(torch.utils.data.Dataset):
    def __init__(self, pairs): self.p = pairs
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return encode(self.p[i])

def collate(feats):
    L = max(len(f["input_ids"]) for f in feats)
    def pad(x, v): return [r + [v] * (L - len(r)) for r in x]
    return {"input_ids": torch.tensor(pad([f["input_ids"] for f in feats], tok.pad_token_id)),
            "labels": torch.tensor(pad([f["labels"] for f in feats], -100)),
            "attention_mask": torch.tensor(pad([f["attention_mask"] for f in feats], 0))}

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, device_map="auto", quantization_config=bnb,
                                             low_cpu_mem_usage=True)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir="/kaggle/working/lora_ckpt", num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=2, gradient_accumulation_steps=8, fp16=True,
    logging_steps=25, eval_strategy="steps", eval_steps=200, save_strategy="no",
    report_to="none", warmup_ratio=0.03, lr_scheduler_type="cosine",
    gradient_checkpointing=True, optim="paged_adamw_8bit")
trainer = Trainer(model=model, args=args, train_dataset=DS(train_pairs),
                  eval_dataset=DS(val_pairs), data_collator=collate)
trainer.train()
model.save_pretrained("/kaggle/working/lora_adapter")
tok.save_pretrained("/kaggle/working/lora_adapter")
print("adapter saved -> /kaggle/working/lora_adapter  (create a Kaggle dataset from this output)")


In [ ]:
# quick sanity: verifier accuracy on the val slice via Yes/No logits
model.eval()
yes_id = tok.encode(" Yes", add_special_tokens=False)[0]
no_id  = tok.encode(" No",  add_special_tokens=False)[0]
good = tot = 0
for user, ans in val_pairs[:200]:
    prompt = tok.apply_chat_template([{"role": "system", "content": SYS},
                                      {"role": "user", "content": user}],
                                     tokenize=False, add_generation_prompt=True)
    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=MAXLEN - 4).to(model.device)
    with torch.no_grad():
        lg = model(**enc).logits[0, -1]
    pred = 1 if lg[yes_id] > lg[no_id] else 0
    good += (pred == (1 if ans.strip() == "Yes" else 0)); tot += 1
print(f"val verifier accuracy: {good}/{tot} = {good/tot:.3f}  "
      "(expect >0.85 on the synthetic-dominated val slice)")
